# Lecture 6 · Notebook 1 — Reward models and DPO from scratch
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture06/lecture06_1_reward_models_and_dpo.ipynb)

**Companion to [Chapters 2–3 of the video lecture](https://nlp-course-cuhksz.github.io/lecture06.html#p2-s22)** (slides 22–24 and 30–37). RLHF turns human comparisons into a
**reward model** with the Bradley–Terry loss, then optimizes the policy against it with a KL leash. DPO skips the reward model and
trains the policy directly on the same preference pairs. In this notebook you will:

1. see why annotators' **scores** disagree while their **comparisons** agree, and fit a Bradley–Terry reward model (slide 22),
2. watch a biased reward model get **hacked** by best-of-N selection: the proxy reward keeps rising, true quality does not (slide 37),
3. check the key identity behind DPO on a tiny "bandit": the policy DPO learns is exactly the RLHF optimum
   $\pi^*(y) \propto \pi_\text{ref}(y)\,e^{r(y)/\beta}$ (slide 30), and
4. write the **DPO loss** for a real language model (GPT-2) and train it on a handful of preference pairs (slide 31).

Runs on CPU in about 3–5 minutes (sections 1–3 take seconds; section 4 downloads GPT-2, about 550 MB). No GPU needed.

In [ ]:
!pip -q install -U transformers

In [ ]:
import math, random
import numpy as np
import torch, torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0); random.seed(0)

## 1 · Scores are noisy, comparisons are not
Forty summaries have a hidden true quality $q$. Three annotators score them on a 1–10 scale, but each has a private **scale** (how
spread out their scores are), a private **offset** (strict or generous) and some noise. This is the situation on slide 22: the same
summary can get 8, 5 and 9.

In [ ]:
n_items = 40
q = np.random.randn(n_items)                                   # hidden true quality
annotators = {"strict":   (1.0, 3.5), "generous": (1.5, 7.5), "moody": (2.0, 5.5)}   # (scale, offset)

def score(name, i, noise=0.6):
    a, b = annotators[name]
    return float(np.clip(a * q[i] + b + noise * np.random.randn(), 1, 10))

scores = {k: np.array([score(k, i) for i in range(n_items)]) for k in annotators}
for i in range(3):
    print(f"summary {i}:", "  ".join(f"{k}={scores[k][i]:.1f}" for k in annotators))

# How often do two annotators give (almost) the same score, versus the same verdict on a pair?
def same_score(k1, k2): return np.mean(np.abs(scores[k1] - scores[k2]) < 1.0)
def same_verdict(k1, k2, n_pairs=2000):
    agree = 0
    for _ in range(n_pairs):
        i, j = np.random.choice(n_items, 2, replace=False)
        agree += (score(k1, i) > score(k1, j)) == (score(k2, i) > score(k2, j))
    return agree / n_pairs
for k1, k2 in [("strict", "generous"), ("strict", "moody"), ("generous", "moody")]:
    print(f"{k1:8s} vs {k2:8s}: scores within 1 point {same_score(k1, k2):4.0%}   same verdict on a pair {same_verdict(k1, k2):4.0%}")

Absolute scores rarely match, yet the annotators usually agree on **which of two summaries is better**. So we collect comparisons and fit
the **Bradley–Terry** model: $P(y_w \succ y_l) = \sigma\big(r(y_w) - r(y_l)\big)$. The loss is $-\log\sigma(r_w - r_l)$, the same loss
as in panel 2 of slide 23. Only *differences* of rewards appear, so every annotator's private offset drops out.

Here the reward model is the simplest possible one: one learnable number per summary.

In [ ]:
comparisons = []                                              # (winner, loser) pairs from all three annotators
for _ in range(1500):
    k = random.choice(list(annotators)); i, j = np.random.choice(n_items, 2, replace=False)
    comparisons.append((i, j) if score(k, i) > score(k, j) else (j, i))
W = torch.tensor([c[0] for c in comparisons]); L = torch.tensor([c[1] for c in comparisons])

r = torch.zeros(n_items, requires_grad=True)
opt = torch.optim.Adam([r], lr=0.05)
for step in range(500):
    loss = -F.logsigmoid(r[W] - r[L]).mean() + 1e-3 * (r ** 2).mean()    # Bradley-Terry loss (+ tiny L2)
    opt.zero_grad(); loss.backward(); opt.step()
print("final BT loss:", round(loss.item(), 3))

def spearman(x, y):
    rx, ry = np.argsort(np.argsort(x)), np.argsort(np.argsort(y)); return np.corrcoef(rx, ry)[0, 1]
r_hat = r.detach().numpy()
print("rank correlation with true quality:  BT reward = %.3f" % spearman(r_hat, q))
for k in annotators:
    print(f"                                     {k:8s} scores = {spearman(scores[k], q):.3f}")
plt.figure(figsize=(4.5, 3.5)); plt.scatter(q, r_hat, s=15); plt.xlabel("true quality q"); plt.ylabel("learned reward r")
plt.title("Bradley-Terry recovers the ranking"); plt.show()

The learned rewards line up with the true quality (up to a shift and a scale, which BT cannot see), because the model pools the
comparisons of all annotators. A real reward model replaces the free number per summary with a neural network that reads the text,
usually initialized from the language model itself (slide 21).

## 2 · Reward hacking: a biased reward model meets an optimizer
Now suppose annotators have a **length bias** (the "beauty bias" of slide 36: longer, nicer-looking answers win). Each response has a
content quality $c$ and a length $\ell$ (both standardized; length has a heavier tail, since a few answers ramble a lot). The **true**
utility likes content and dislikes rambling; the labellers add a bonus for length:

$$u_\text{true} = c - 0.5\,\max(0, \ell)^2, \qquad u_\text{labeller} = u_\text{true} + 1.0\,\ell .$$

We fit a small reward model, linear in $c$ and $\ell$, on the labellers' comparisons. Like every real reward model it is an imperfect
proxy: it cannot see that *very* long answers get worse. Then we **optimize against it** with the simplest optimizer there is,
best-of-$N$: sample $N$ responses, keep the one the reward model likes best.

In [ ]:
def true_u(c, l):  return c - 0.5 * np.maximum(0, l) ** 2
def labeller_u(c, l): return true_u(c, l) + 1.0 * l
def feats(c, l):   return np.stack([c, l], -1)                  # the reward model's features
def sample_len(*shape): return np.random.laplace(0, 1 / np.sqrt(2), shape)   # unit variance, heavier tail

# 4000 labelled comparisons between random responses
c1, c2 = np.random.randn(2, 4000); l1, l2 = sample_len(4000), sample_len(4000)
pref = (labeller_u(c1, l1) + 0.3 * np.random.randn(4000) > labeller_u(c2, l2)).astype(np.float32)
X = torch.tensor(feats(c1, l1) - feats(c2, l2), dtype=torch.float32)   # BT on a linear RM = logistic regression on differences
w = torch.zeros(2, requires_grad=True); opt = torch.optim.Adam([w], lr=0.05)
for _ in range(400):
    loss = F.binary_cross_entropy_with_logits(X @ w, torch.tensor(pref)); opt.zero_grad(); loss.backward(); opt.step()
w_np = w.detach().numpy()
print("reward model weights [content, length]:", np.round(w_np, 2))
def rm(c, l): return feats(c, l) @ w_np

Ns = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
proxy, true = [], []
for N in Ns:
    c, l = np.random.randn(2000, N), sample_len(2000, N)      # 2000 prompts, N candidates each
    best = rm(c, l).argmax(1)
    cb, lb = np.take_along_axis(c, best[:, None], 1)[:, 0], np.take_along_axis(l, best[:, None], 1)[:, 0]
    proxy.append(rm(cb, lb).mean()); true.append(true_u(cb, lb).mean())
kl = [math.log(N) - (N - 1) / N for N in Ns]                 # KL(best-of-N || base policy), a standard closed form
fig, ax = plt.subplots(figsize=(6, 3.8))
ax.plot(kl, proxy, "o-", label="reward-model score (proxy)"); ax.plot(kl, true, "s-", label="true utility")
for k_, N, t in zip(kl, Ns, true): ax.annotate(f"N={N}", (k_, t), fontsize=7, xytext=(2, -10), textcoords="offset points")
ax.set_xlabel("KL from the starting policy (nats)"); ax.legend(); ax.set_title("Goodhart's law with best-of-N"); plt.show()
print("true utility peaks at N =", Ns[int(np.argmax(true))])

The proxy score keeps climbing while the true utility peaks and then falls: the optimizer has found the reward model's soft spot (length)
and is exploiting it. That is **over-optimization**, the first symptom on slide 37. Two remedies from the lecture: keep the policy close to
the reference model (the $\beta\,\mathrm{KL}$ leash on slide 24; on this plot, stay on the left), or use a reward that is **checked**
instead of learned (RLVR, Chapter 5).

## 3 · DPO on a bandit: no reward model, same optimum
Take one prompt with six possible answers. The reference policy $\pi_\text{ref}$ and the true rewards $r$ are fixed. RLHF with a KL
penalty $\beta$ has a closed-form optimum (slide 30):

$$\pi^*(y) = \frac{1}{Z}\,\pi_\text{ref}(y)\,\exp\big(r(y)/\beta\big).$$

DPO never builds a reward model. It only sees preference pairs $(y_w, y_l)$ sampled from Bradley–Terry with the true $r$, and minimizes

$$\mathcal{L}_\text{DPO} = -\log\sigma\Big(\beta\log\frac{\pi_\theta(y_w)}{\pi_\text{ref}(y_w)} - \beta\log\frac{\pi_\theta(y_l)}{\pi_\text{ref}(y_l)}\Big).$$

If the identity on slide 30 is right, the DPO policy should land on $\pi^*$.

In [ ]:
answers = ["ideal", "good", "ok", "verbose", "wrong", "harmful"]
r_true  = torch.tensor([2.0, 1.5, 0.5, 0.0, -1.0, -3.0])
pi_ref  = torch.tensor([0.10, 0.15, 0.30, 0.25, 0.15, 0.05])      # the SFT model likes "ok" and "verbose"
beta = 0.5
pi_star = pi_ref * torch.exp(r_true / beta); pi_star /= pi_star.sum()

# preference data: random pairs, labelled by Bradley-Terry with the true reward
i = torch.randint(0, 6, (5000,)); j = torch.randint(0, 6, (5000,)); keep = i != j; i, j = i[keep], j[keep]
i_wins = torch.rand(len(i)) < torch.sigmoid(r_true[i] - r_true[j])
yw, yl = torch.where(i_wins, i, j), torch.where(i_wins, j, i)

theta = torch.log(pi_ref).clone().requires_grad_(True)          # policy logits, initialised at the reference
opt = torch.optim.Adam([theta], lr=0.05)
for step in range(1500):
    logp = F.log_softmax(theta, -1)
    margin = beta * ((logp[yw] - torch.log(pi_ref[yw])) - (logp[yl] - torch.log(pi_ref[yl])))
    loss = -F.logsigmoid(margin).mean()
    opt.zero_grad(); loss.backward(); opt.step()
pi_dpo = F.softmax(theta, -1).detach()
implicit_r = beta * torch.log(pi_dpo / pi_ref)                  # "your language model is secretly a reward model"
print(f"{'answer':8s} {'pi_ref':>7s} {'pi*':>7s} {'pi_DPO':>7s} {'r':>6s} {'implicit r (shifted)':>21s}")
for a, p0, ps, pd, rt, ri in zip(answers, pi_ref, pi_star, pi_dpo, r_true, implicit_r - implicit_r.mean() + r_true.mean()):
    print(f"{a:8s} {p0:7.3f} {ps:7.3f} {pd:7.3f} {rt:6.2f} {ri:21.2f}")

DPO recovers $\pi^*$ (up to sampling noise in the 5000 pairs), and $\beta\log\frac{\pi_\theta}{\pi_\text{ref}}$ recovers the reward up
to a constant, without ever fitting a reward model or sampling from the policy. Try `beta = 0.1` and `beta = 5`: a small $\beta$ is a
long leash (the policy piles onto the best answer), a large $\beta$ keeps it close to $\pi_\text{ref}$.

## 4 · The DPO loss for a real language model
For a language model, $\log\pi_\theta(y\mid x)$ is the sum of the log-probabilities of the **answer tokens** given the prompt; the
prompt tokens are masked out, exactly as in SFT (slide 6). The policy is GPT-2 (124M); the reference is a frozen copy. The preference
pairs below are small versions of the examples in Chapter 2: helpful, honest and harmless answers versus unhelpful, made-up or
harmful ones.

In [ ]:
import copy
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained("gpt2")
policy = AutoModelForCausalLM.from_pretrained("gpt2")
ref = copy.deepcopy(policy).eval().requires_grad_(False)
print(f"GPT-2: {sum(p.numel() for p in policy.parameters()) / 1e6:.0f}M parameters")

pairs = [  # (prompt, chosen, rejected)
    ("My cat hasn't eaten since yesterday. What should I do?",
     " Not eating for a day can be serious in cats. Please call a vet today.", " Just wait a week, cats often skip meals."),
    ("Explain the moon landing to a 6 year old.",
     " Some brave people flew a big rocket to the Moon, walked on it, and came back home.", " Explain the theory of gravity to a 6 year old."),
    ("Who was the first person to walk on Mars?",
     " Nobody has walked on Mars yet. Only robots have been there.", " Neil Armstrong walked on Mars in 1969."),
    ("How do I kill a stuck Python process?",
     " Find its process ID and end it, for example with the kill command.", " Sorry, I cannot help with killing anything."),
    ("How can I get into my neighbour's Wi-Fi without the password?",
     " I can't help with that, but you could ask your neighbour or set up your own plan.", " Use a password cracking tool on their router."),
    ("Tell me how Columbus felt when he came to the US in 2015.",
     " Columbus died in 1506, so he could not visit in 2015, but we can imagine it.", " Columbus was very excited to be in the US in 2015."),
    ("Give me a reference for the claim that coffee cures cancer.",
     " I don't know of good evidence that coffee cures cancer, so I can't cite one.", " See Smith et al., Nature Medicine 2019, page 12."),
    ("I got a KeyError in Python. What does it mean?",
     " Your dictionary has no such key. Check the spelling, or use dict.get with a default.", " A KeyError is an error."),
]
def fmt(p): return f"Question: {p}\nAnswer:"

def seq_logp(model, prompts, answers):
    # sum of log p(answer tokens | prompt) for each example, prompt tokens masked out
    ids, masks = [], []
    for p, a in zip(prompts, answers):
        pi, ai = tok(fmt(p))["input_ids"], tok(a)["input_ids"]
        ids.append(pi + ai); masks.append([0] * len(pi) + [1] * len(ai))
    T = max(map(len, ids))
    x = torch.full((len(ids), T), tok.eos_token_id); m = torch.zeros(len(ids), T); att = torch.zeros(len(ids), T, dtype=torch.long)
    for k, (s, mk) in enumerate(zip(ids, masks)):
        x[k, :len(s)] = torch.tensor(s); m[k, :len(s)] = torch.tensor(mk, dtype=torch.float); att[k, :len(s)] = 1
    logits = model(input_ids=x, attention_mask=att).logits[:, :-1]
    lp = torch.log_softmax(logits.float(), -1).gather(-1, x[:, 1:, None])[..., 0]
    return (lp * m[:, 1:]).sum(-1)

def dpo_loss(beta=0.1):
    P, C, R = zip(*pairs)
    pc, pr = seq_logp(policy, P, C), seq_logp(policy, P, R)
    with torch.no_grad():
        rc, rr = seq_logp(ref, P, C), seq_logp(ref, P, R)
    chosen_reward, rejected_reward = beta * (pc - rc), beta * (pr - rr)     # implicit rewards
    loss = -F.logsigmoid(chosen_reward - rejected_reward).mean()
    return loss, chosen_reward.detach(), rejected_reward.detach(), pc.detach(), pr.detach()

loss, cr, rr_, _, _ = dpo_loss()
print(f"DPO loss before training: {loss.item():.4f}   (log 2 = {math.log(2):.4f}, because policy == reference)")

At the start the policy *is* the reference, so every implicit reward is 0 and the loss is exactly $\log 2$. Now train for a few steps.

In [ ]:
opt = torch.optim.AdamW(policy.parameters(), lr=1e-5)
log = []
# (we keep dropout off, i.e. eval mode, so that policy and reference agree exactly at step 0)
for step in range(25):
    loss, cr, rj, pc, pr = dpo_loss()
    opt.zero_grad(); loss.backward(); opt.step()
    log.append((loss.item(), (cr > rj).float().mean().item(), (cr - rj).mean().item(), pc.mean().item(), pr.mean().item()))
    if step % 5 == 0 or step == 24:
        print(f"step {step:2d}  loss {log[-1][0]:.3f}  pair accuracy {log[-1][1]:.2f}  reward margin {log[-1][2]:+.2f}")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot([l[0] for l in log]); ax[0].axhline(math.log(2), ls=":", c="gray"); ax[0].set_title("DPO loss"); ax[0].set_xlabel("step")
ax[1].plot([l[3] for l in log], label="log p(chosen)"); ax[1].plot([l[4] for l in log], label="log p(rejected)")
ax[1].set_title("sequence log-probs under the policy"); ax[1].set_xlabel("step"); ax[1].legend(); plt.show()

The loss falls and the **margin** between chosen and rejected grows. Look at the right panel: DPO only constrains the *difference*, so
nothing forces the chosen answers up. In this tiny run (8 pairs) the chosen answers do rise while the rejected ones fall, but on real
datasets it is common to see both fall, the rejected ones faster. This is one reason people add an SFT term on the chosen answers, or move to iterative / on-policy methods
(slide 31).

Finally, the implicit reward of each pair after training:

In [ ]:
with torch.no_grad():
    _, cr, rj, _, _ = dpo_loss()
for (p, c, r_), a, b in zip(pairs, cr, rj):
    print(f"{a.item():+6.2f} vs {b.item():+6.2f}   {p[:55]}")

Eight pairs and 25 steps only *memorize* these preferences; real DPO uses tens of thousands of pairs (UltraFeedback has about 60K), but
the loss is the one you just wrote.

## Exercises
1. **Section 1:** give the "moody" annotator twice as much noise. Which degrades faster: agreement on scores or on verdicts? Does the BT
   reward still rank the summaries well?
2. **Section 2:** set the labellers' length bonus to 0. Does the true utility still peak? Then give the reward model the feature
   `l ** 2` as well. Which fix helps more at $N = 512$: unbiased labels or a better reward model?
3. **Section 3:** train with `beta = 0.1` and `beta = 5` and plot $\pi_\text{DPO}$ against $\pi_\text{ref}$. Then label the pairs
   with a *wrong* reward (swap "ideal" and "harmful"): DPO learns the wrong thing just as faithfully. Garbage preferences in, garbage policy
   out.
4. **Section 4:** add `0.1 * (-pc.mean())` (an SFT term on the chosen answers) to the DPO loss. What happens to `log p(chosen)`?

**Next:** [Notebook 2 — GRPO advantages and pass@k on a verifiable task](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture06/lecture06_2_grpo_and_pass_at_k.ipynb)